# Acquirium client reference

This is the reference guide for the acquirium client: the query interface feature by feature, with the internals (`show_query_graph`, `to_sparql`, text resolution) shown along the way. If you are new, start with `quickstart.ipynb`.

To get support reach out to [Mete](mailto:saka@mines.edu)!

## Initial setup:
1. create and activate a fresh virtual environment: `python -m venv .venv && source .venv/bin/activate` (Linux/macOS) or `.venv\Scripts\activate` (Windows). **Requires Python 3.12**+.
    - Alternatively you can use [uv package manager] with `uv init --python 3.12`
2. Install Acquirium from PyPI: `pip install acquirium[watertap]`.
    - Alternatively: `uv add acquirium[watertap]`
3. Start the server (plus any drivers listed in the config): 
    - `acquirium server --config deployments/WATERTAP/acquirium.toml`.
    - Alternatively, `uv run acquirium server --config deployments/WATERTAP/acquirium.toml`.
4. Verify it's up by opening [`http://localhost:8000/docs`](http://localhost:8000/docs) (or whichever host/port your config sets) in a browser.
    - Alternatively: `curl localhost:8000/health` from another terminal
    - Or using Python session or notebook, run:
    ```
    from acquirium import Acquirium 
    acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)
    ```

## Loading required libraries and acquirium client

Acquirium is shipped with it's client to connect the server and help users to use the server with a python interface.

Acquirium client can be loaded and initiated with:

In [82]:
from datetime import datetime, timedelta, timezone
from acquirium import Acquirium

acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)

## Find entities by class
`_class` accepts a URI or a natural-language string. `alias` names the node for later reference.

In [83]:
q = acq.find_entity(_class="Pump", alias="pump")
_ = q.metadata_head()

Metadata First
   10 Rows    
┏━━━━━━━━━━━━┓
┃ pump       ┃
┡━━━━━━━━━━━━┩
│ wbs:P1     │
│ wbs:P2     │
│ wbs:intake │
└────────────┘

### How strings become URIs
Every string is resolved server-side by an embedding matcher. `resolve_text` shows what a string resolves to — check it when a query returns something unexpected, and pass exact URIs when correctness matters (the top match is not always the intended one):

In [ ]:
acq.client.resolve_text("salt", kind="class", top_k=3)

## Follow relationships
`find_related` adds a neighbour reachable within `hops`. `predicates` restricts which edges to follow (`multi_hop_predicates=True` applies them at every hop); `direction="upstream"/"downstream"` walks the S223 piping topology instead. Strings and URIs are both accepted.

In [84]:
q = (
    acq.find_entity(_class="Pump", alias="pump")
       .find_related(_class="Tank", alias="tank", _from="pump", hops=1)
)
q.show_query_graph()
_ = q.metadata_head()

QUERY GRAPH

Nodes:
  0 [pump]  class=http://data.ashrae.org/standard223#Pump
  2 [tank]  class=urn:nawi-water-ontology#Tank

Edges:
  pump --(*, hops=1)--> tank

Data nodes: (none)

Current pointer: tank



           Metadata First 10 Rows            
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pump       ┃ tank                         ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:intake │ wbs:ferric-chloride-addition │
└────────────┴──────────────────────────────┘

## Attach data nodes
`find_data` adds the observable/actuatable properties of the current node. `find_all_data` does it for every entity in the graph.


In [85]:
q = acq.find_entity(_class="Pump", alias="pump").find_data()
_ = q.metadata_head()

             Metadata First 10 Rows             
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pump       ┃ pump_data                       ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:P1     │ wbs:P1-out-pressure             │
│ wbs:P1     │ wbs:P1-out-pressure             │
│ wbs:intake │ wbs:intake-in-tss-concentration │
│ wbs:intake │ wbs:intake-in-tss-concentration │
│ wbs:intake │ wbs:intake-in-flow-rate         │
│ wbs:intake │ wbs:intake-in-flow-rate         │
│ wbs:intake │ wbs:intake-in-tds-concentration │
│ wbs:intake │ wbs:intake-in-tds-concentration │
│ wbs:P1     │ wbs:P1-efficiency               │
│ wbs:P1     │ wbs:P1-efficiency               │
└────────────┴─────────────────────────────────┘

In [86]:
q_all = acq.find_all_data()
_ =q_all.metadata_head()

             Metadata First 10 Rows             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:storage-tank-3-out-flow-rate             │
│ wbs:RO-out-flow-mass-water                   │
│ wbs:P1-out-pressure                          │
│ wbs:conn-cartridge-filtration-to-S1-pressure │
│ wbs:intake-in-tds-concentration              │
│ wbs:PXR-brine-out-flow-mass-tds              │
│ wbs:P1-mechanical-power                      │
│ wbs:intake-in-tds-concentration              │
│ wbs:storage-tank-3-out-flow-rate             │
│ wbs:RO-out-retentate-flow-mass-tds           │
└──────────────────────────────────────────────┘

## Filter data nodes
Filters apply to the bound data nodes. Strings are resolved via the text matcher.

In [87]:
q = (
    acq.find_all_data()
       .filter_by_quantity_kind("Pressure")
)
_ =q.metadata_head()

             Metadata First 10 Rows             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:conn-cartridge-filtration-to-S1-pressure │
│ wbs:conn-cartridge-filtration-to-S1-pressure │
│ wbs:PXR-brine-out-pressure                   │
│ wbs:PXR-brine-out-pressure                   │
│ wbs:RO-in-pressure                           │
│ wbs:RO-in-pressure                           │
│ wbs:RO-out-retentate-pressure                │
│ wbs:RO-out-retentate-pressure                │
│ wbs:RO-out-pressure                          │
│ wbs:RO-out-pressure                          │
└──────────────────────────────────────────────┘

In [88]:
q = (
    acq.find_all_data()
       .filter_by_unit("KG/s")
)
_ =q.metadata_head()

                Metadata First 10 Rows                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                                   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:PXR-brine-out-flow-mass-water                   │
│ wbs:PXR-brine-out-flow-mass-water                   │
│ wbs:RO-in-flow-mass-water                           │
│ wbs:RO-in-flow-mass-water                           │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-water │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-water │
│ wbs:RO-out-flow-mass-tds                            │
│ wbs:RO-out-flow-mass-tds                            │
│ wbs:RO-out-retentate-flow-mass-tds                  │
│ wbs:RO-out-retentate-flow-mass-tds                  │
└─────────────────────────────────────────────────────┘

In [89]:
q = (
    acq.find_all_data()
        .filter_by_substance("constituent Salt")
        .filter_by_unit("KG/s")
)
_ =q.metadata_head()

               Metadata First 10 Rows                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                                 ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:RO-out-flow-mass-tds                          │
│ wbs:RO-out-flow-mass-tds                          │
│ wbs:RO-out-retentate-flow-mass-tds                │
│ wbs:RO-out-retentate-flow-mass-tds                │
│ wbs:RO-in-flow-mass-tds                           │
│ wbs:RO-in-flow-mass-tds                           │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-tds │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-tds │
│ wbs:PXR-brine-out-flow-mass-tds                   │
│ wbs:PXR-brine-out-flow-mass-tds                   │
└───────────────────────────────────────────────────┘

The built-in filters map to fixed predicates (`qudt:hasUnit`, `s223:ofSubstance`, `qudt:hasQuantityKind`, `s223:hasMedium`). For any other predicate use `filter_data_nodes` directly (values are exact URIs, no text resolution):

## Inspect the query
`show_query_graph` prints the node/edge structure; `to_sparql` returns the compiled SPARQL; `metadata` returns the full result as a polars DataFrame.

In [90]:
q.show_query_graph()

QUERY GRAPH

Nodes:
  0 [0] [DATA]  class=*

Edges:

Data nodes:
  0 [0]  filters={http://data.ashrae.org/standard223#ofSubstance=['urn:nawi-water-ontology#Constituent-Salt'], http://qudt.org/schema/qudt/hasUnit=['http://qudt.org/vocab/unit/KiloGM-PER-SEC']}}

Current pointer: 0



In [91]:
print(q.to_sparql())

SELECT DISTINCT ?v0 ?ext0 ?unit0 ?extunit0
WHERE {
  ?v0 <https://brickschema.org/schema/Brick/ref#hasExternalReference> ?ext0 .
  OPTIONAL { ?v0 <http://qudt.org/schema/qudt/hasUnit> ?unit0 . }
  OPTIONAL { ?ext0 <http://qudt.org/schema/qudt/hasUnit> ?extunit0 . }
  { { ?v0 <http://data.ashrae.org/standard223#ofSubstance> <urn:nawi-water-ontology#Constituent-Salt> . } }
  { { ?v0 <http://qudt.org/schema/qudt/hasUnit> <http://qudt.org/vocab/unit/KiloGM-PER-SEC> . } }
}


In [92]:
df_meta = q.metadata()
df_meta

0
str
"""wbs:RO-out-flow-mass-tds"""
"""wbs:RO-out-flow-mass-tds"""
"""wbs:RO-out-retentate-flow-mass…"
"""wbs:RO-out-retentate-flow-mass…"
"""wbs:RO-in-flow-mass-tds"""
"""wbs:RO-in-flow-mass-tds"""
"""wbs:conn-cartridge-filtration-…"
"""wbs:conn-cartridge-filtration-…"
"""wbs:PXR-brine-out-flow-mass-td…"


## Pull timeseries
`dataframe` returns a polars frame. `shape="wide"` puts each data node in its own column, `"narrow"` is long-form. `latest_data` is a shortcut for the most recent point.

In [93]:
end = datetime.now(tz=timezone.utc)
start = end - timedelta(minutes=10)

df = q.dataframe(start=start, end=end, shape="wide", cast_value="float")
df.head()

time,wbs:PXR-brine-out-flow-mass-tds,wbs:RO-out-retentate-flow-mass-tds,wbs:RO-in-flow-mass-tds,wbs:RO-out-flow-mass-tds,wbs:conn-cartridge-filtration-to-S1-flow-mass-tds
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-07-01 17:44:22.305945 UTC,11.936148,11.936148,11.965374,0.029226,11.965374
2026-07-01 17:44:38.227281 UTC,12.085548,12.085548,12.114765,0.029217,12.114765
2026-07-01 17:44:58.919918 UTC,11.825058,11.825058,11.854187,0.029129,11.854187
2026-07-01 17:45:14.203376 UTC,11.672406,11.672406,11.701795,0.029389,11.701795
2026-07-01 17:45:31.152047 UTC,12.035707,12.035707,12.064624,0.028917,12.064624


In [94]:
q.latest_data(limit=2)

time,wbs:RO-out-flow-mass-tds,wbs:PXR-brine-out-flow-mass-tds,wbs:RO-out-retentate-flow-mass-tds,wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,wbs:RO-in-flow-mass-tds
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-07-01 00:16:36.510997 UTC,0.027768,12.39511,12.39511,12.422878,12.422878
2026-07-01 00:16:43.839118 UTC,0.028225,11.589517,11.589517,11.617742,11.617742
2026-07-01 17:53:24.833440 UTC,0.02929,null,11.32193,11.35122,11.35122
2026-07-01 17:53:43.428754 UTC,0.029283,11.881243,11.881243,11.910526,11.910526
2026-07-01 17:54:12.980023 UTC,null,11.749138,null,null,null


## Structured access via DataObject
`data()` returns an object keyed by alias for quick lookups.

In [95]:
import polars as pl


data = q.data(start=start, end=end, cast_value="float")
data.dataframe().cast(pl.Float64)

time,0__wbs:RO-out-retentate-flow-mass-tds,0__wbs:RO-in-flow-mass-tds,0__wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,0__wbs:PXR-brine-out-flow-mass-tds,0__wbs:RO-out-flow-mass-tds
f64,f64,f64,f64,f64,f64
1.7829e15,11.936148,11.965374,11.965374,11.936148,0.029226
1.7829e15,12.085548,12.114765,12.114765,12.085548,0.029217
1.7829e15,11.825058,11.854187,11.854187,11.825058,0.029129
1.7829e15,11.672406,11.701795,11.701795,11.672406,0.029389
1.7829e15,12.035707,12.064624,12.064624,12.035707,0.028917
…,…,…,…,…,…
1.7829e15,12.119786,12.149017,12.149017,12.119786,0.02923
1.7829e15,12.154375,12.18372,12.18372,12.154375,0.029344
1.7829e15,11.537435,11.566725,11.566725,11.537435,0.029289


## Inspect units
`units()` returns the effective QUDT unit URI per data alias.

In [96]:
data.units()

{'0': 'http://qudt.org/vocab/unit/KiloGM-PER-SEC'}

## Convert units
`convert_to(target)` accepts any QUDT-recognized identifier (URI, label, symbol, UCUM code). The returned DataObject has values converted and `units()` updated.

In [97]:
data = data.convert_to("kg/min")
data.dataframe().head()

time,0__wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,0__wbs:RO-out-retentate-flow-mass-tds,0__wbs:RO-in-flow-mass-tds,0__wbs:PXR-brine-out-flow-mass-tds,0__wbs:RO-out-flow-mass-tds
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-07-01 17:44:22.305945 UTC,717.922456,716.168877,717.922456,716.168877,1.753579
2026-07-01 17:44:38.227281 UTC,726.885889,725.132873,726.885889,725.132873,1.753017
2026-07-01 17:44:58.919918 UTC,711.251221,709.50348,711.251221,709.50348,1.747741
2026-07-01 17:45:14.203376 UTC,702.107687,700.344351,702.107687,700.344351,1.763337
2026-07-01 17:45:31.152047 UTC,723.877415,722.142407,723.877415,722.142407,1.735008


### Systems

Systems are logical groupings of equipment and junctions (and other systems) in S223 ontology (parent ontology of WaTr)

The systems in the model are:

In [98]:
def list_systems():
    q = acq.find_entity(_class = "System", alias = "Systems")
    return q.metadata()
list_systems()

Systems
str
"""wbs:pretreatment-system"""
"""wbs:desalination-system"""
"""wbs:posttreatment-system"""
"""wbs:seawater-ro-plant"""


The systems are hierarchically organized as:

In [99]:
def list_systems_hier():
    q = acq.find_entity(_class = "System", alias = "Systems")
    q = q.find_related(_class = "System", predicates = ['hasMember'], alias = "Subsystem")
    q.metadata_head()
list_systems_hier()

               Metadata First 10 Rows               
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Systems               ┃ Subsystem                ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:seawater-ro-plant │ wbs:pretreatment-system  │
│ wbs:seawater-ro-plant │ wbs:desalination-system  │
│ wbs:seawater-ro-plant │ wbs:posttreatment-system │
└───────────────────────┴──────────────────────────┘

We can see how many equipment we have in each system:

In [100]:
def list_equipment_by_system(hops = 1):
        q = acq.find_entity(_class = "System", alias = "Systems")
        q = q.find_related(_class = "Equipment", predicates = ['hasMember'], alias = "Equipment", hops = hops, multi_hop_predicates = True)
        q_df = q.metadata()
        return q_df.group_by("Systems").agg(pl.col("Equipment").count().alias("equipment_count")).sort("equipment_count", descending=True)

list_equipment_by_system()

Systems,equipment_count
str,u32
"""wbs:pretreatment-system""",9
"""wbs:posttreatment-system""",5
"""wbs:desalination-system""",4


These are the number of equipment directly a member of these systems.

If we increase the hops, you'll see total number equipments in each system

In [101]:
list_equipment_by_system(3)

Systems,equipment_count
str,u32
"""wbs:seawater-ro-plant""",18
"""wbs:pretreatment-system""",9
"""wbs:posttreatment-system""",5
"""wbs:desalination-system""",4


Let's find the pumps in a specific system:


In [102]:
def list_equipment_in_system(system, equipment):
    q = acq.find_entity(uri=system, alias = "system").find_related(_class=equipment, alias = "equipment", hops=1)
    q.metadata_head()

list_equipment_in_system('wbs:pretreatment-system', 'pump')

         Metadata First 10 Rows         
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ system                  ┃ equipment  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ wbs:pretreatment-system │ wbs:intake │
└─────────────────────────┴────────────┘

Let's find all the pumps and their data

In [103]:
def all_pumps_and_their_data():
    q = acq.find_entity(_class="pump", alias="pump").find_all_data()
    q.metadata_head()
    return q.data(limit=10).dataframe()

all_pumps_and_their_data().head()

             Metadata First 10 Rows             
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pump       ┃ pump_data                       ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:P1     │ wbs:P1-out-pressure             │
│ wbs:P1     │ wbs:P1-out-pressure             │
│ wbs:intake │ wbs:intake-in-tss-concentration │
│ wbs:intake │ wbs:intake-in-tss-concentration │
│ wbs:intake │ wbs:intake-in-flow-rate         │
│ wbs:intake │ wbs:intake-in-flow-rate         │
│ wbs:intake │ wbs:intake-in-tds-concentration │
│ wbs:intake │ wbs:intake-in-tds-concentration │
│ wbs:P1     │ wbs:P1-efficiency               │
│ wbs:P1     │ wbs:P1-efficiency               │
└────────────┴─────────────────────────────────┘

time,pump_data__wbs:P2-efficiency,pump_data__wbs:intake-in-tds-concentration,pump_data__wbs:intake-in-flow-rate,pump_data__wbs:P1-out-pressure,pump_data__wbs:P1-efficiency,pump_data__wbs:P2-mechanical-power,pump_data__wbs:intake-in-tss-concentration,pump_data__wbs:P1-mechanical-power
"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,f64,f64
2026-07-01 00:13:18.140063 UTC,0.8,33.699458,0.341333,7e6,0.8,149579.616304,0.038317,1.2561e6
2026-07-01 00:13:27.210202 UTC,0.8,33.699458,0.341333,7e6,0.8,147512.002088,0.038317,1.2406e6
2026-07-01 00:13:33.844473 UTC,0.8,33.699458,0.341333,7e6,0.8,146752.494201,0.038317,1.2332e6
2026-07-01 00:15:20.114227 UTC,0.8,33.699458,0.351572,7e6,0.8,156542.546383,0.038317,1.2468e6
2026-07-01 00:16:15.624634 UTC,0.8,33.699458,0.354985,7e6,0.8,159902.69015,0.038317,1.2512e6


Let's find all the data generating entites within a system:

In [104]:
def find_all_sensors(system):
    q = (acq.find_entity(uri=system, alias="backwash")
         .find_related(_class="equipment", alias="equipment",predicates=['hasMember'], hops=1)
         .find_data(alias = "sensors"))
    q_df = q.metadata(include_internals=True)
    q_df = q_df.drop([pl.col('backwash'),pl.col('sensors_ref'),pl.col('extunit4')])
    return q_df

system = 'wbs:pretreatment-system'
find_all_sensors(system)

equipment,sensors,unit4
str,str,str
"""wbs:intake""","""wbs:intake-in-tss-concentratio…","""unit:MilliGM-PER-L"""
"""wbs:intake""","""wbs:intake-in-tss-concentratio…","""unit:MilliGM-PER-L"""
"""wbs:intake""","""wbs:intake-in-flow-rate""","""None"""
"""wbs:intake""","""wbs:intake-in-flow-rate""","""None"""
"""wbs:intake""","""wbs:intake-in-tds-concentratio…","""unit:KiloGM-PER-M3"""
"""wbs:intake""","""wbs:intake-in-tds-concentratio…","""unit:KiloGM-PER-M3"""
